# Stability
1. Select cell types and samples from each tissue type that have a large number of cells.
3. Generate 10 70% subsamples of the RNA-seq and ATAC-seq datasets for each of the 10 sample cell types
4. Use the corresponding trained holdout model to generate cross-evaluation GRNs for each of the 10 subsamples
5. Generate GRNs for each of the 10 subsamples using the other GRN inference methods
6. Calculate the average pairwise Jaccard index for the 10 GRNs generated by my models and the other GRN inference methods

In [51]:
%matplotlib inline
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64,garbage_collection_threshold:0.6"

import sys
from pathlib import Path
from collections import Counter
import muon as mu
import logging
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import mudata
mudata.set_options(pull_on_update=False)
from matplotlib_inline import backend_inline

PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/TETHER")
sys.path.append(str(PROJECT_DIR))

DATA_DIR = PROJECT_DIR.parent / "data"
CHKPT_DIR = PROJECT_DIR / "checkpoints"

PROJECT_DATA_DIR = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER")

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

BINDING_MANIFEST = "binding_manifest.json"

backend_inline.set_matplotlib_formats('svg')

In [ ]:
species = "mm10"

PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/TETHER")
DATA_DIR = PROJECT_DIR.parent / "data"

# tissue type, dataset, sample name, cell type
sample_sources = [
    ("HSC", "GSE246464_HSC", "young_rep1", "HSC"),
    ("Liver", "mouse_liver", "liver_sample", "Hepatocytes"), # mouse_4
    ("Brain", "10x_E18_mouse_brain", "E18_brain", "Excitatory_neurons"),
    ("Embryo", "mESC", "E7.5_rep2", "Neurectoderm"),
    ("Kidney", "kidney", "Ctrl_4weeks_2", "Fibroblasts"),
    ("Skin", "GSE140203_shareseq", "skin_late_anagen", "Basal_keratinocytes"),
]

In [63]:
import json

percent_subsample = 0.7
num_subsamples = 10

subsample_cell_dict = {}

json_subsample_file = PROJECT_DATA_DIR / "subsample_cell_dict.json"

np.random.default_rng(42)
if json_subsample_file.exists():
    with open(json_subsample_file, "r") as f:
        subsample_cell_dict = json.load(f)
else:
    for tissue, dataset, sample_name, cell_type in sample_sources:
        input_data_dir = DATA_DIR / "processed" / dataset / sample_name
        assert input_data_dir.exists(), f"Input data directory does not exist: {input_data_dir}"
        
        subsample_cell_dict[tissue] = {
            "dataset": dataset,
            "cell_type": cell_type,
            "sample_name": sample_name,
            "subsamples": {}
        }

        mdata = mu.read(input_data_dir / "multiome_processed.h5mu", backed="r")
        try:
            rna_obs = mdata.mod["rna"].obs
            mask = rna_obs["celltype"].astype(str).str.lower().str.contains(cell_type.lower(), regex=False)
            if tissue == "Liver":
                mask &= rna_obs["donor"].astype(str) == "mouse_4"

            mdata_celltype = mdata[rna_obs.index[mask.to_numpy()]]
            n_cells = mdata_celltype.n_obs
            
            n_random_cells = int(n_cells *percent_subsample)
            
            # Randomly select cells for the subsamples
            for i in range(num_subsamples):
                cell_sample_ids = np.random.choice(mdata_celltype.obs_names, size=n_random_cells, replace=False)
                
                subsample_cell_dict[tissue]["subsamples"][f"subsample_{i}"] = list(cell_sample_ids)
        
        finally:
            mdata.file.close()
            
        with open(json_subsample_file, "w") as f:
            json.dump(subsample_cell_dict, f, indent=4)

